In [ ]:
# CELL 0: CÀI ĐẶT THƯ VIỆN VÀ KẾT NỐI GOOGLE DRIVE
!pip install -q -U ultralytics transformers accelerate timm supervision addict yapf scikit-learn

import os, sys, glob, random, shutil, copy, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device)
print("PyTorch:", torch.__version__)

In [ ]:
# CELL 1: CHIA DỮ LIỆU SẠCH 70/15/15
src_dataset = '/content/drive/MyDrive/AnhTest/archive/dataset'
data_path = '/content/drive/MyDrive/AnhTest/VN_Real_Food_Split'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SPLIT_RATIOS = (0.70, 0.15, 0.15)

if not os.path.isdir(src_dataset):
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu nguồn: {src_dataset}")

class_dirs = sorted([p for p in Path(src_dataset).iterdir() if p.is_dir()])
if len(class_dirs) != 10:
    print(f"CẢNH BÁO: tìm thấy {len(class_dirs)} lớp, outline yêu cầu 10 lớp.")

# Xóa toàn bộ thư mục đích cũ để không trộn dữ liệu các lần chạy
if os.path.exists(data_path):
    shutil.rmtree(data_path)

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(data_path, split), exist_ok=True)

split_summary = {}
for class_dir in class_dirs:
    images = sorted([p for p in class_dir.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    random.Random(SEED).shuffle(images)

    n = len(images)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val = int(n * SPLIT_RATIOS[1])
    # Phần còn lại đưa vào Test để tổng số ảnh được bảo toàn
    groups = {
        'train': images[:n_train],
        'val': images[n_train:n_train+n_val],
        'test': images[n_train+n_val:]
    }

    for split, files_list in groups.items():
        target_dir = os.path.join(data_path, split, class_dir.name)
        os.makedirs(target_dir, exist_ok=True)
        for src in files_list:
            shutil.copy2(src, os.path.join(target_dir, src.name))

    split_summary[class_dir.name] = {k: len(v) for k, v in groups.items()}

summary_df = pd.DataFrame(split_summary).T
summary_df.loc['TỔNG'] = summary_df.sum(numeric_only=True)
display(summary_df)
print("\nĐã chia dữ liệu tại:", data_path)

In [ ]:
# CELL 2: KIỂM TRA CẤU TRÚC, SỐ ẢNH VÀ TỶ LỆ CHIA
def count_images(folder):
    return sum(1 for p in Path(folder).rglob('*')
               if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

for split in ['train', 'val', 'test']:
    print(f"{split.upper():>5}: {count_images(os.path.join(data_path, split))} ảnh")

print("\nSố lớp trong từng tập:")
for split in ['train', 'val', 'test']:
    split_dir = Path(data_path) / split
    class_count = len([p for p in split_dir.iterdir() if p.is_dir()])
    print(split, ":", class_count, "lớp")

assert all((Path(data_path) / s).is_dir() for s in ['train', 'val', 'test'])
assert count_images(Path(data_path) / 'test') > 0, "Tập Test đang rỗng!"
print("\nResize mục tiêu: 448x448 cho YOLO; ảnh được resize trong pipeline của mô hình.")
print("Chuẩn hóa pixel và augmentation được thực hiện trong pipeline huấn luyện.")

In [ ]:
# CELL 3: HUẤN LUYỆN YOLOv8s-cls VỚI CALLBACK GIẢM LEARNING RATE
from ultralytics import YOLO
import torch

yolo_model = YOLO('yolov8s-cls.pt')

# Callback ReduceLROnPlateau theo top-1 validation accuracy
scheduler_state = {'scheduler': None}

def create_plateau_scheduler(trainer):
    scheduler_state['scheduler'] = torch.optim.lr_scheduler.ReduceLROnPlateau(
        trainer.optimizer, mode='max', factor=0.5, patience=3,
        threshold=1e-4, min_lr=1e-6
    )

def step_plateau_scheduler(trainer):
    scheduler = scheduler_state.get('scheduler')
    if scheduler is None:
        return
    metrics = getattr(trainer, 'metrics', {}) or {}
    metric_value = metrics.get('metrics/accuracy_top1')
    if metric_value is not None:
        scheduler.step(float(metric_value))

yolo_model.add_callback('on_train_start', create_plateau_scheduler)
yolo_model.add_callback('on_fit_epoch_end', step_plateau_scheduler)

yolo_results = yolo_model.train(
    data=data_path,
    epochs=100,
    imgsz=448,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    freeze=10,
    patience=12,             # EarlyStopping của Ultralytics
    auto_augment='randaugment',
    erasing=0.4,
    fliplr=0.5,
    flipud=0.0,
    workers=2,
    seed=SEED,
    deterministic=True,
    project='runs/classify',
    name='vn_foods_yolov8s',
    exist_ok=True,
    plots=True,
    verbose=True
)

YOLO_BEST_PATH = str(Path(yolo_model.trainer.save_dir) / 'weights' / 'best.pt')
print("Đường dẫn trọng số tốt nhất:", YOLO_BEST_PATH)

In [ ]:
# CELL 4: ĐỒ THỊ LOSS VÀ ACCURACY CỦA YOLO
results_csv = Path(yolo_model.trainer.save_dir) / 'results.csv'
if results_csv.exists():
    yolo_df = pd.read_csv(results_csv)
    yolo_df.columns = yolo_df.columns.str.strip()
    print("Các cột kết quả:", yolo_df.columns.tolist())

    loss_cols = [c for c in yolo_df.columns if 'loss' in c.lower()]
    acc_cols = [c for c in yolo_df.columns if 'accuracy' in c.lower() or 'top1' in c.lower()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col in loss_cols:
        axes[0].plot(yolo_df.index + 1, yolo_df[col], label=col)
    axes[0].set_title('Loss theo Epoch')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    if loss_cols: axes[0].legend(); axes[0].grid(True)

    for col in acc_cols:
        axes[1].plot(yolo_df.index + 1, yolo_df[col], label=col)
    axes[1].set_title('Accuracy theo Epoch')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    if acc_cols: axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig('training_curves_yolo.png', dpi=150)
    plt.show()
else:
    print("Chưa tìm thấy results.csv. Hãy chạy xong Cell 3 trước.")

In [ ]:
# CELL 5: HUẤN LUYỆN CUSTOM CNN
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import copy

IMG_SIZE_CNN = 224
BATCH_CNN = 32
cnn_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
cnn_eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(data_path, 'train'), transform=cnn_train_tf)
val_ds = datasets.ImageFolder(os.path.join(data_path, 'val'), transform=cnn_eval_tf)
test_ds = datasets.ImageFolder(os.path.join(data_path, 'test'), transform=cnn_eval_tf)
class_names = train_ds.classes
assert train_ds.class_to_idx == val_ds.class_to_idx == test_ds.class_to_idx, "Thứ tự lớp giữa các tập không đồng nhất!"

train_loader = DataLoader(train_ds, batch_size=BATCH_CNN, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_CNN, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_CNN, shuffle=False, num_workers=2, pin_memory=True)

class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

cnn_model = CustomCNN(len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
cnn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
)

def run_epoch(model, loader, training=False):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

history_cnn = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc, best_weights, wait = 0.0, copy.deepcopy(cnn_model.state_dict()), 0
EPOCHS_CNN, EARLY_PATIENCE = 40, 8

for epoch in range(EPOCHS_CNN):
    tr_loss, tr_acc = run_epoch(cnn_model, train_loader, training=True)
    va_loss, va_acc = run_epoch(cnn_model, val_loader, training=False)
    cnn_scheduler.step(va_acc)

    history_cnn['train_loss'].append(tr_loss); history_cnn['val_loss'].append(va_loss)
    history_cnn['train_acc'].append(tr_acc); history_cnn['val_acc'].append(va_acc)
    print(f"Epoch {epoch+1:02d}/{EPOCHS_CNN} | train loss={tr_loss:.4f}, acc={tr_acc:.4f} | val loss={va_loss:.4f}, acc={va_acc:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_weights = copy.deepcopy(cnn_model.state_dict())
        wait = 0
        torch.save(best_weights, 'best_custom_cnn.pth')
    else:
        wait += 1
        if wait >= EARLY_PATIENCE:
            print("EarlyStopping: Validation Accuracy không cải thiện.")
            break

cnn_model.load_state_dict(best_weights)
print("Best Validation Accuracy Custom CNN:", best_val_acc)
print("Đã lưu best_custom_cnn.pth")

In [ ]:
# CELL 6: ĐỒ THỊ LOSS / ACCURACY CUSTOM CNN
epochs_axis = range(1, len(history_cnn['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_axis, history_cnn['train_loss'], label='Train Loss')
axes[0].plot(epochs_axis, history_cnn['val_loss'], label='Validation Loss')
axes[0].set_title('Custom CNN - Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(epochs_axis, history_cnn['train_acc'], label='Train Accuracy')
axes[1].plot(epochs_axis, history_cnn['val_acc'], label='Validation Accuracy')
axes[1].set_title('Custom CNN - Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig('training_curves_custom_cnn.png', dpi=150)
plt.show()

In [ ]:
# CELL 7: CLASSIFICATION REPORT + CONFUSION MATRIX CHO CUSTOM CNN
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

def evaluate_torch_model(model, loader, class_names, title):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            y_pred.extend(outputs.argmax(1).cpu().numpy().tolist())
            y_true.extend(labels.numpy().tolist())

    print(title)
    print("Test Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 10))
    disp.plot(ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False)
    plt.title(title + ' - Confusion Matrix')
    plt.tight_layout()
    plt.savefig('confusion_matrix_custom_cnn_test.png', dpi=150)
    plt.show()
    return accuracy_score(y_true, y_pred)

cnn_test_accuracy = evaluate_torch_model(cnn_model, test_loader, class_names, 'Custom CNN')

In [ ]:
# CELL 8: ĐÁNH GIÁ YOLO TRÊN CÙNG TẬP TEST
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from ultralytics import YOLO

yolo_best_candidates = glob.glob('runs/classify/vn_foods_yolov8s*/weights/best.pt')
if not yolo_best_candidates:
    # fallback nếu Colab chạy tiếp từ phiên khác
    yolo_best_candidates = glob.glob('runs/classify/*/weights/best.pt')
if not yolo_best_candidates:
    raise FileNotFoundError("Không tìm thấy best.pt. Hãy chạy huấn luyện YOLO trước.")
YOLO_BEST_PATH = sorted(yolo_best_candidates, key=os.path.getmtime)[-1]
yolo_test_model = YOLO(YOLO_BEST_PATH)

test_root = Path(data_path) / 'test'
y_true_yolo, y_pred_yolo = [], []
for class_name in sorted([p.name for p in test_root.iterdir() if p.is_dir()]):
    class_folder = test_root / class_name
    image_files = [str(p) for p in class_folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    for image_file in image_files:
        pred_result = yolo_test_model.predict(image_file, verbose=False)[0]
        pred_idx = int(pred_result.probs.top1)
        y_pred_yolo.append(pred_idx)
        y_true_yolo.append(class_names.index(class_name))

print("YOLOv8s-cls - Test Accuracy:", round(accuracy_score(y_true_yolo, y_pred_yolo), 4))
print(classification_report(y_true_yolo, y_pred_yolo, target_names=class_names, zero_division=0))
cm_yolo = confusion_matrix(y_true_yolo, y_pred_yolo)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm_yolo, display_labels=class_names).plot(ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False)
plt.title('YOLOv8s-cls - Confusion Matrix trên Test')
plt.tight_layout()
plt.savefig('confusion_matrix_yolo_test.png', dpi=150)
plt.show()
yolo_test_accuracy = accuracy_score(y_true_yolo, y_pred_yolo)

In [ ]:
# CELL 9: BẢNG SO SÁNH HAI MÔ HÌNH
comparison_df = pd.DataFrame([
    {'Mô hình': 'YOLOv8s-cls (Transfer Learning)', 'Test Accuracy': yolo_test_accuracy},
    {'Mô hình': 'Custom CNN', 'Test Accuracy': cnn_test_accuracy}
])
display(comparison_df)
comparison_df.to_csv('model_comparison_test.csv', index=False)
print("So sánh công bằng: cùng tập Test và cùng cách tính Top-1 Accuracy.")

In [ ]:
# CELL 10: KIỂM THỬ ẢNH MỜ, THIẾU SÁNG, GÓC LỆCH VÀ CHE KHUẤT
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw
from google.colab import files
from IPython.display import display

uploaded_edge = files.upload()
if uploaded_edge:
    edge_path = next(iter(uploaded_edge))
    original = Image.open(edge_path).convert('RGB')

    # Che khuất mô phỏng bằng một hình chữ nhật trên một phần ảnh
    occluded = original.copy()
    draw_occ = ImageDraw.Draw(occluded)
    ow, oh = occluded.size
    draw_occ.rectangle([ow*0.35, oh*0.35, ow*0.65, oh*0.65], fill=(30, 30, 30))

    edge_images = {
        'Ảnh gốc': original,
        'Thiếu sáng': ImageEnhance.Brightness(original).enhance(0.35),
        'Quá sáng': ImageEnhance.Brightness(original).enhance(1.7),
        'Ảnh mờ': original.filter(ImageFilter.GaussianBlur(radius=4)),
        'Góc lệch 25 độ': original.rotate(25, expand=True),
        'Che khuất mô phỏng': occluded
    }

    edge_model = YOLO(YOLO_BEST_PATH)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, (label, pil_img) in zip(axes.flatten(), edge_images.items()):
        temp_path = f'/content/edge_{len(label)}.jpg'
        pil_img.save(temp_path)
        pred = edge_model.predict(temp_path, verbose=False)[0]
        top_idx = int(pred.probs.top1)
        top_name = pred.names[top_idx]
        confidence = float(pred.probs.top1conf)
        ax.imshow(pil_img)
        ax.set_title(f'{label}\n{top_name}: {confidence:.2%}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Bạn chưa tải ảnh lên; bỏ qua kiểm thử Edge Cases.")

In [ ]:
# CELL 11: DEMO PHÂN LOẠI MÓN ĂN TRÊN VIDEO
import cv2
from google.colab import files
from IPython.display import Video, display

uploaded_video = files.upload()
if uploaded_video:
    video_input = next(iter(uploaded_video))
    cap = cv2.VideoCapture(video_input)
    if not cap.isOpened():
        raise RuntimeError("Không mở được video đã tải lên.")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0: fps = 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_path = 'food_detection_video.mp4'
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    video_model = YOLO(YOLO_BEST_PATH)
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        result = video_model.predict(frame, verbose=False)[0]
        pred_idx = int(result.probs.top1)
        pred_name = result.names[pred_idx]
        confidence = float(result.probs.top1conf)
        cv2.putText(frame, f'{pred_name}: {confidence:.1%}', (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        writer.write(frame)
        frame_count += 1

    cap.release()
    writer.release()
    print(f"Đã xử lý {frame_count} khung hình. Video kết quả: {output_path}")
    display(Video(output_path, embed=True))
else:
    print("Bạn chưa tải video lên; bỏ qua demo video.")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torchvision.ops as ops
from PIL import Image, ImageDraw, ImageFont
import glob
from google.colab import files
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# 1. TẢI MÔ HÌNH VÀO GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ Đang tải mô hình lên {device}...")

# Tải YOLO (Lấy kết quả train mới nhất)
best_candidates = glob.glob('runs/classify/vn_foods_yolov8s*/weights/best.pt')
if not best_candidates:
    best_candidates = glob.glob('runs/classify/*/weights/best.pt')
best_path_for_demo = sorted(best_candidates, key=os.path.getmtime)[-1] if best_candidates else None
model_food = YOLO(best_path_for_demo) if best_path_for_demo else None

# Tải Grounding DINO
dino_id = "IDEA-Research/grounding-dino-base"
processor = AutoProcessor.from_pretrained(dino_id)
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)
print("✅ Nạp mô hình hoàn tất!\n")

# Bảng Prompt RÚT GỌN (Giúp DINO tự tin hơn, ra điểm % cao hơn)
TOPPING_PROMPTS = {
    'banh_trung_thu': "salted egg yolk . purple filling . baked pastry crust . sliced mooncake .",
    'banh_mi': "cucumber . cilantro . pork sausage . fried egg . red chili . pate .",
    'banh_xeo': "cooked shrimp . pork slice . bean sprouts . crispy pancake crust . scallion .",
    'banh_khot': "cooked shrimp . scallion oil . mini crispy pancake .",
    'banh_bot_loc': "cooked shrimp . pork belly . clear chewy dough .",
    'banh_da_lon': "green pandan layer . yellow mung bean layer .",
    'banh_bo': "steamed sponge cake . honeycomb texture .",
    'banh_cong': "crispy cup cake . fried whole shrimp .",
    'banh_tieu': "fried puffed bread . white sesame seed .",
    'banh_tai_heo': "spiral swirl biscuit ."
}

TRANSLATION_MAP = {
    'salted egg yolk': 'Lòng đỏ trứng muối',
    'egg yolk': 'Lòng đỏ trứng',
    'purple filling': 'Nhân khoai lang tím',
    'baked pastry crust': 'Vỏ bánh nướng',
    'pastry crust': 'Vỏ bánh',
    'sliced mooncake': 'Miếng bánh',
    'mooncake': 'Bánh trung thu',
    'cucumber': 'Dưa leo',
    'cilantro': 'Ngò rí',
    'pork sausage': 'Chả lụa',
    'fried egg': 'Trứng ốp la',
    'red chili': 'Ớt đỏ',
    'pate': 'Pa-tê',
    'cooked shrimp': 'Tôm chín',
    'shrimp': 'Tôm',
    'pork slice': 'Thịt thái lát',
    'bean sprouts': 'Giá đỗ',
    'crispy pancake crust': 'Vỏ bánh xèo',
    'mini crispy pancake': 'Bánh khọt',
    'scallion oil': 'Mỡ hành',
    'scallion': 'Hành lá'
}

def translate_label(raw_label):
    raw_label = raw_label.strip().lower()
    if raw_label in TRANSLATION_MAP:
        return TRANSLATION_MAP[raw_label]
    for eng_key, vie_val in TRANSLATION_MAP.items():
        if eng_key in raw_label:
            return vie_val
    return raw_label.capitalize()

# 2. XỬ LÝ ẢNH
print("📤 Hãy chọn ảnh món ăn từ máy tính để test (hoặc bấm Hủy để dùng ảnh mẫu):")
uploaded = files.upload()

img_path = next(iter(uploaded)) if uploaded else next((f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.png', '.jpeg'))), None)

if not img_path:
    print("❌ Không tìm thấy bức ảnh nào để phân tích!")
else:
    print(f"\n🔍 Đang phân tích ảnh: {img_path}")
    raw_image = Image.open(img_path).convert("RGB")
    w_orig, h_orig = raw_image.size

    # --- STAGE 1: DỰ ĐOÁN MÓN VỚI YOLO ---
    detected_dish = "banh_trung_thu"
    print("="*65)
    if model_food:
        res_food = model_food.predict(img_path, verbose=False)
        detected_dish = res_food[0].names[res_food[0].probs.top1]
        print(f"🍽️ [Stage 1 - YOLO]: TÊN MÓN LÀ **{detected_dish.upper().replace('_', ' ')}** ({res_food[0].probs.top1conf.item() * 100:.2f}%)")
    else:
        print(f"⚠️ Chưa có mô hình YOLO (best.pt). Mặc định giả định món: **{detected_dish.upper().replace('_', ' ')}**")

    # --- STAGE 2: BÓC TÁCH BẰNG GROUNDING DINO ---
    prompt = TOPPING_PROMPTS.get(detected_dish, "food . meat . egg . vegetable . sauce .")
    inputs = processor(images=raw_image, text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = grounding_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=0.25, text_threshold=0.20, target_sizes=[(h_orig, w_orig)]
    )[0]

    raw_boxes, raw_scores, raw_labels = results["boxes"], results["scores"], results["labels"]

    if len(raw_boxes) > 0:
        # LỌC TRÙNG LẶP (NMS) VÀ SẮP XẾP CAO XUỐNG THẤP
        keep_indices = ops.nms(raw_boxes, raw_scores, iou_threshold=0.45).cpu().numpy()
        filtered_results = [
            (raw_scores[i].item(), raw_labels[i], raw_boxes[i].cpu().tolist())
            for i in keep_indices
        ]
        filtered_results.sort(key=lambda x: x[0], reverse=True)

        # VẼ KHUNG VÀ CHỮ LÊN ẢNH
        draw_img = raw_image.copy()
        draw = ImageDraw.Draw(draw_img)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 16)
            font_large = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 22)
        except:
            font = font_large = ImageFont.load_default()

        print("-" * 65)
        print("📋 DANH SÁCH THÀNH PHẦN (ĐÃ SẮP XẾP THEO ĐỘ TIN CẬY):")
        print("-" * 65)

        for idx, (score, label, box) in enumerate(filtered_results, start=1):
            x1, y1, x2, y2 = [int(v) for v in box]
            vn_label = translate_label(label)

            print(f" {idx:02d}. {vn_label:<25} | Tin cậy: {score*100:.2f}% | Toạ độ: [{x1}, {y1}, {x2}, {y2}]")

            draw.rectangle([x1, y1, x2, y2], outline="#00FF00", width=3)
            display_text = f"#{idx} {vn_label} {score*100:.1f}%"
            text_bbox = draw.textbbox((0, 0), display_text, font=font)
            text_width, text_height = text_bbox[2] - text_bbox[0], text_bbox[3] - text_bbox[1]

            label_y1 = max(0, y1 - text_height - 6)
            draw.rectangle([x1, label_y1, x1 + text_width + 6, label_y1 + text_height + 6], fill="#00AA00")
            draw.text((x1 + 3, label_y1 + 3), display_text, fill="white", font=font)

        # In tên món ăn lớn ở góc trái trên cùng
        dish_title = f" MÓN: {detected_dish.upper().replace('_', ' ')} "
        title_bbox = draw.textbbox((0, 0), dish_title, font=font_large)
        draw.rectangle([0, 0, title_bbox[2] + 10, title_bbox[3] + 10], fill="#FF0000")
        draw.text((5, 5), dish_title, fill="white", font=font_large)

        out_file = "final_output.jpg"
        draw_img.save(out_file)
        print("=" * 65)
        print("\n🖼️ ẢNH KẾT QUẢ ĐÃ TỐI ƯU HÓA:")
        display(IPyImage(out_file, width=500))
    else:
        print("⚠️ Không phát hiện thành phần nào đạt ngưỡng tin cậy.")
        display(raw_image)